# Smoke tests: SmolVLA and Evo-1 on a Kaggle T4

Plan item 1.6. Each model trains 100 steps on the public dataset `lerobot/svla_so100_pickplace` to check the environment, the model download and one optimisation loop. The results say nothing about task performance.

Settings: GPU T4 x2 (one GPU is used), internet on. Both models run in one environment with LeRobot at commit `e581c1d` (Python 3.12, installed with uv). The T4 (Turing) supports neither bfloat16 nor flash attention, so Evo-1 runs in float16 without them.

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
LEROBOT = "lerobot[training,evo1,smolvla] @ git+https://github.com/huggingface/lerobot@e581c1d5701f846788662079d4ac08835c96def9"


In [ ]:
%%bash
set -e
pip install -q uv
uv venv -q /kaggle/working/venv --python 3.12
uv pip install -q --python /kaggle/working/venv/bin/python "lerobot[training,evo1,smolvla] @ git+https://github.com/huggingface/lerobot@e581c1d5701f846788662079d4ac08835c96def9"
/kaggle/working/venv/bin/python -c "import torch, lerobot, transformers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'lerobot', lerobot.__version__, 'transformers', transformers.__version__)"


In [ ]:
# Peak GPU memory is sampled with nvidia-smi while each run trains.
import json, re, threading, time, pathlib

VENV = "/kaggle/working/venv/bin"
OUT = pathlib.Path("/kaggle/working/smoke")
OUT.mkdir(exist_ok=True)
pathlib.Path("/kaggle/working/evo1_train_sdpa.py").write_text("""
import lerobot.policies.evo1.internvl3_embedder as emb
from lerobot.scripts.lerobot_train import main
_load = emb.AutoModel.from_pretrained
def _sdpa(*a, **k):
    if k.get("attn_implementation") == "eager":
        k["attn_implementation"] = "sdpa"
    return _load(*a, **k)
emb.AutoModel.from_pretrained = _sdpa
main()
""")

def run(name, cmd):
    peak = [0]
    stop = threading.Event()
    def sample():
        while not stop.is_set():
            used = subprocess.run(["nvidia-smi", "--id=0", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
            if used.isdigit():
                peak[0] = max(peak[0], int(used))
            time.sleep(1)
    thread = threading.Thread(target=sample, daemon=True)
    thread.start()
    start = time.time()
    log = OUT / f"{name}.log"
    with log.open("w") as sink:
        code = subprocess.run(cmd, stdout=sink, stderr=subprocess.STDOUT, env={**__import__("os").environ, "CUDA_VISIBLE_DEVICES": "0"}).returncode
    stop.set()
    text = log.read_text(errors="replace")
    steps = re.findall(r"step:(\d+)\S* .*?loss:([\d.]+)", text)
    result = {"run": name, "exit": code, "wall_s": round(time.time() - start), "peak_gpu_mib": peak[0],
              "last_step": int(steps[-1][0]) if steps else 0, "last_loss": float(steps[-1][1]) if steps else None}
    print(json.dumps(result))
    if code:
        print("\n".join(text.splitlines()[-25:]))
    return result

COMMON = ["--dataset.repo_id=lerobot/svla_so100_pickplace", "--steps=100", "--log_freq=10",
          "--num_workers=2", "--save_checkpoint=false", "--policy.push_to_hub=false", "--policy.device=cuda"]
RENAME = '--rename_map={"observation.images.top": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}'
results = []


In [ ]:
results.append(run("smolvla_b8", [f"{VENV}/lerobot-train", "--policy.path=lerobot/smolvla_base", RENAME,
    "--policy.empty_cameras=1", "--batch_size=8", "--output_dir=/kaggle/working/smoke/smolvla_b8", *COMMON]))

In [ ]:
EVO1 = ["--policy.type=evo1", "--policy.dtype=float16", "--policy.use_amp=false", "--policy.use_flash_attn=false"]
results.append(run("evo1_b8_eager", [f"{VENV}/lerobot-train", *EVO1, "--batch_size=8",
    "--output_dir=/kaggle/working/smoke/evo1_b8_eager", *COMMON]))
if results[-1]["exit"]:
    results.append(run("evo1_b8_sdpa", [f"{VENV}/python", "/kaggle/working/evo1_train_sdpa.py", *EVO1,
        "--batch_size=8", "--output_dir=/kaggle/working/smoke/evo1_b8_sdpa", *COMMON]))

In [ ]:
(OUT / "results.json").write_text(json.dumps(results, indent=2))
for r in results:
    print(r)